## Gemini

Push VQA to Google Gemini.

### Docs:

* in theory, there is documentation here: https://ai.google.dev/gemini-api/docs, 
* Here though, I just asked claude again :D

#### Key Points (cp-claude)

* API Key: Get your API key from the [Google Console](https://aistudio.google.com/app/apikey) -- note that you either have to create a new GCP project OR create the API key in an existing project
* Then the rest of the steps is similar for Claude/ChatGPT


First, install anthropic api (also, see .yml file for the environment for this project)

In [1]:
#!pip install google-genai

Where are things stored/going to be stored?

What is thinking budget:

| Model | Min | Max | Can disable (=0)? |
|---|---|---|---|
| `gemini-2.5-flash` | 1 | 24,576 | ✅ yes |
| `gemini-2.5-flash-lite` | 512 | 24,576 | ✅ yes |
| `gemini-2.5-pro` | 128 | 32,768 | ❌ no |

Setting `thinking_budget` to `-1` turns on dynamic thinking, where the model automatically adjusts the budget based on the complexity of the request. Google AI When dynamic thinking is enabled this way, the default max is 8,192 tokens.

In [2]:
dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini/' #store API results for example data
model_name='gemini-3.1-flash-lite-preview' # test 
#model_name='gemini-2.5-flash' # test 
#thinking_budget = 0 # 0 for minimal thinking
thinking_budget = -1 # will change based on task

key_file = '/Users/jnaiman/.gemini/key.txt'

# where is VQA dataset?
jsons_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/' # directory where jsons created with figure are stored
imgs_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/' # where images are stored

# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/' # this might not be used...

img_format = 'jpeg'

# for asking for reasoning
reasoning_text = 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.'

In [3]:
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob
# import google.generativeai as genai
from google import genai
from google.genai import types

# debug
from importlib import reload
from copy import deepcopy


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

In [4]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

# Configure the API key
#genai.configure(api_key=api_key.strip())
client = genai.Client(api_key=api_key.strip())

config=types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget)  # disable for speed/cost
)

In [5]:
#model_gemini = genai.GenerativeModel(model)

In [6]:
jsons_to_parse = glob(jsons_dir + '/*.json')
jsons_to_parse[:3]

['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000015_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000005_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000077_qa.json']

In [7]:
def send_to_gemini(question_list, image_path, client,model_name='gemini-3.1-flash-lite-preview',
                    test_run = True, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, img_format='jpg',         
                   large_image= False, config=None, reasoning=None):
    """
    Sends an image + question to Gemini.  Does something different for large file, but might be bad idea.

    system_prompt : set to None to use default from question list
    """
    if system_prompt is None:
        system_prompt = question_list['persona']
    if img_format.lower() == 'jpg':
        img_format = 'jpeg'

    err = False
    #model_gemini = genai.GenerativeModel(model)
    #if verbose: print('Model loaded:', model)
    prompt_save = ''; prompt = ''; response = ''
    if not test_run:
        question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
        if reasoning is not None:
            question += " " + reasoning
        # lowercase the first word, just in case
        question = question.lstrip() # no whitespace
        question = question[0].lower() + question[1:]

        if verbose: print('   on question:',question)
        # Prepare the API request
        prompt = f"I am going to show you an image. Now, {question}"
        prompt_save = f"I am going to show you an image. Now, {question}"
        try:
        #if True:

            # # Wait for processing (for video files, you might need to wait longer)
            # if large_image:
            #     # Upload the file to Gemini
            #     #uploaded_file = genai.upload_file(path=image_path)
            #     uploaded_file = client.files.upload(file=image_path)
            #     import time
            #     while uploaded_file.state.name == "PROCESSING":
            #         time.sleep(1)
            #         uploaded_file = genai.get_file(uploaded_file.name)
            # else:
            #     uploaded_file = Image.open(image_path)

            with open(image_path, 'rb') as f:
                image_bytes = f.read()
            image_part = types.Part.from_bytes(data=image_bytes, mime_type='image/'+img_format)

            
            if config is None:
                config = types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    # other config here
                )
            # Generate content with the uploaded file
            #response = model_gemini.generate_content([prompt, uploaded_file])
            response = client.models.generate_content(
                model=model_name,
                contents=[prompt, image_part],
                config=config
            )
            
            if large_image:
                # Clean up - delete the uploaded file
                #genai.delete_file(uploaded_file.name)
                client.files.delete(name=uploaded_file.name)
            
            #return response.text
        
        except Exception as e:
            print('[ERROR]:', str(e))
            err = True

            #return f"Error: {str(e)}"
        
    if not test_run and not err:
        for part in response.candidates[0].content.parts:
            if not part.thought:  # skip thinking tokens
                text = part.text
        if verbose:
            #print('     response:', str(response.candidates[0].content.parts[0].text).replace('\n',''))
            print('     response:', text)
        question_list['Response (raw)'] = deepcopy(response)
        # Get the response from the API
        #answer = deepcopy(str(response.candidates[0].content.parts[0].text))
        answer = text
        question_list['raw answer'] = answer
        # also calculate usage
        usage = {
            'input_tokens': response.usage_metadata.prompt_token_count,
            'output_tokens': response.usage_metadata.candidates_token_count,
            'total_tokens': response.usage_metadata.total_token_count,
            'cached_content_tokens': getattr(response.usage_metadata, 'cached_content_token_count', 0)
            }

        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        if '```json"' in answer:
            answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        elif '```json\n' in answer:
            answer_format = answer.split('```json\n')[-1].split('\n')[0].replace('\n', '')
        elif '```json' in answer:
            answer_format = answer.split('```json')[-1].split('\n')[0].replace('\n', '')
        else:
            answer_format = answer # just give it a shot!
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
    elif err:
        question_list['raw answer'] = 'ERROR'
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['Response (raw)'] = 'TEST RUN'
    

    return question_list, prompt_save, system_prompt

In [ ]:
reload(utils.llm_utils)
from utils.llm_utils import parse_for_errors

iMax = 10
verbose = True
test_run = False # run w/o actually pinging openai
restart = False
# set system_prompt to None to default to what is in question list
system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
if reasoning_text is not None: # rephrease is reasoning is requested
    system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any text outside of the requested JSON responses."""

#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
# temperature=0.1

fac = 1.0

for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    _, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, 
                                                            dir_api, restart=restart,fac=fac,
                                                      tmp_dir=tmp_dir, 
                                                      img_format=img_format) #, load_image=False)
    if err:
        continue
    if verbose: print('Got image!')

    ###### create QA ########
    qa = []
    
    # start with figure level questions
    for k,v in base_json['VQA']['Level 1']['Figure-level questions'].items():
        out = {'Q':v['Q'], 'A':v['A'], 'Level':'Level 1', 'type':'Figure-level questions', 'Response':"", 
               "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
               "reasoning":reasoning_text}
        qa.append(out)
    for level in ['Level 2', 'Level 3']:
        if level in base_json['VQA']:
            if 'Figure-level questions' in base_json['VQA'][level]:
                #print('** yes, level ***', level)
                for k,v in base_json['VQA'][level]['Figure-level questions'].items():
                    out = {'Q':v['Q'], 'A':v['A'], 'Level':level, 'type':'Figure-level questions', 'Response':"", 
                        "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
                        "reasoning":reasoning_text}
                    qa.append(out)
    
    # what kinds?
    #types = ['(words + list)', '(words)']
    types_qa = []
    
    # get uniques
    level_parse = 'Level 1'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)
    
    level_parse = 'Level 2'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)
    
    level_parse = 'Level 3'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)

    responses = []; prompts = []; system_prompts = []
    for question_list in qa:
        response, prompt, system_prompt_out = send_to_gemini(question_list, img_path, client,
                    test_run = test_run, 
                    verbose=verbose, img_format=img_format,
                    system_prompt = system_prompt, config=config, reasoning = reasoning_text)
        responses.append(response)
        question_list['prompt'] = prompt
        question_list['system prompt'] = system_prompt_out
        #import sys; sys.exit()


    # parse for errors
    print('')
    print('**** Cleaned QA ****')
    qa = parse_for_errors(qa, llm='Gemini')
    #qa = parse_for_errors(qa) # might need to do this again

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model_name], ff)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()

print('!!!!!!!! DONE !!!!!!!!!!')

on 0 of 1
Got image!
   on question: how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
     response: {"nrows": "1", "ncols": "2"}
{"explanation": "The image contains two distinct graphical panels positioned side-by-side. The left panel is a scatter plot with a color scale, and the right panel is a line graph titled 'NATURE RESOLUTION'. Thus, the figure is arranged in 1 row and 2 columns."}
   on question: assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please f

In [9]:
#question_list

## Look at data

Check out one, if you wanna:

In [15]:
pickles = glob(dir_api + '*.pickle')
#pickles = glob('/Users/jnaiman/Downloads/tmp/JCDL2025/example_hists/claude_api/*pickle')
pickles[:5]

['/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini/Picture_000015_qa.pickle']

In [16]:
ifile = 0
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [17]:
qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images.  How many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'A': {'nrows': 1, 'ncols': 2},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': '{"nrows": "1", "ncols": "2"}\n{"explanation": "The image contains two distinct graphical panels positioned side-by-side. The left panel is a scatter plot with a color scale, and the right panel is a line graph titled \'NATURE RESOLUTION\'. Thus, the figure is arranged in 1 row and 2 columns."}',
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': '',
 'question': 'How many panels are in this figure?',
 'format': 'Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'reasoning': 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"expl

In [13]:
#qa_in = parse_for_errors_claude(qa_in, llm='gemini')

Claude outputs reasoning, so we have to do a bit of cleaning from the responses:

In [18]:
print(pickles[ifile])
print('*********')
for qa_pairs in qa_in:
    print('Prompt:', qa_pairs['prompt'])
    print('  Real A:', qa_pairs['A'])
    print('Gemini A:', qa_pairs['raw answer'])
    print('    input tokens:', qa_pairs['usage']['input_tokens'])
    print('    output tokens:', qa_pairs['usage']['output_tokens'])
    print('')

/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini/Picture_000015_qa.pickle
*********
Prompt: I am going to show you an image. Now, how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
  Real A: {'nrows': 1, 'ncols': 2}
Gemini A: {"nrows": "1", "ncols": "2"}
{"explanation": "The image contains two distinct graphical panels positioned side-by-side. The left panel is a scatter plot with a color scale, and the right panel is a line graph titled 'NATURE RESOLUTION'. Thus, the figure is arranged in 1 row and 2 columns."}
    input tokens: 1192
    output tokens: 74

Prompt: I am going to show you an image. Now, assume this i